# AcademicForge - AMD ROCm Backend Setup
This notebook sets up the AcademicForge local backend (Gemma + FastAPI) on an AMD GPU environment and runs the Streamlit UI on Hugging Face's default web port (`7860`).


### 1. Stop and Clean Up Old Instances
Run this cell first to release ports and prevent bind conflicts.


In [ ]:
# Terminate any running servers
!pkill -f uvicorn
!pkill -f streamlit
!pkill -f ngrok


### 2. Install AMD ROCm PyTorch & Dependencies
This cell uninstalls the standard CPU/CUDA PyTorch and installs the correct ROCm version, followed by the project dependencies.


In [ ]:
# 1. Uninstall CPU version of PyTorch
!pip uninstall -y torch

# 2. Install PyTorch with ROCm 6.0 support
!pip install torch --index-url https://download.pytorch.org/whl/rocm6.0

# 3. Install remaining local project requirements
!pip install -r requirements-local.txt


### 3. Configure Authentication Tokens
Define your Hugging Face and Ngrok tokens here. We have pre-filled them with your current credentials.


In [ ]:
# Enter your tokens below
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"


### 4. Start the Backend & Streamlit Frontend
This cell starts the FastAPI backend (on port 8000) and the Streamlit frontend (on port 7860).


In [ ]:
import os
import subprocess
import time

# 1. Configure Hugging Face credentials and local model
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["LOCAL_LLM_PROVIDER"] = "transformers"
os.environ["LOCAL_LLM_MODEL"] = "google/gemma-2-2b-it"

# 2. Start the FastAPI backend on port 8000 (using venv if available)
print("Starting backend...")
python_cmd = "venv/bin/python" if os.path.exists("venv") else "python"
backend = subprocess.Popen([python_cmd, "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(10)  # Wait for uvicorn to start and load weights

# 3. Start the Streamlit frontend on port 7860 (Hugging Face default web port)
print("Starting Streamlit frontend on port 7860...")
frontend = subprocess.Popen([python_cmd, "-m", "streamlit", "run", "frontend/streamlit_app.py", "--server.address", "0.0.0.0", "--server.port", "7860"])
time.sleep(3)

print("
" + "="*60)
print("✅ SUCCESS! Your backend & frontend are active.")
print("👉 OPEN THIS URL: https://hf-308-940cf411.hf.space/")
print("="*60)


### 5. Optional: Expose using Ngrok (Alternative to Space Proxy)
Run this cell only if you want to explicitly tunnel the Streamlit app on port 7860 using Ngrok.


In [ ]:
# Connect ngrok to Streamlit port (7860)
if NGROK_TOKEN and NGROK_TOKEN != "YOUR_NGROK_TOKEN_HERE":
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(7860)
    print(f"🌍 Public Ngrok URL: {public_url}")
else:
    print("Ngrok token not configured or not needed.")
